In [24]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Pedali-Part1-2")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .getOrCreate())
sc = spark.sparkContext

print("Spark version :", spark.version)
print("Master        :", sc.master)
print("appName       :", sc.appName)
print("Coeurs (defaultParallelism) :", sc.defaultParallelism)

Spark version : 3.5.3
Master        : spark://spark-master:7077
appName       : Pedali-Part1-2
Coeurs (defaultParallelism) : 4


In [25]:
# Création d'un RDD


import json, time
from operator import add

# 1) Paralléliser une collection
nums = sc.parallelize([1, 2, 3, 4, 5, 6], 3)
print("parallelize :", nums.getNumPartitions(), "partitions")

# 2) Charger un fichier externe (1 ligne = 1 élément)
raw = sc.textFile("file:///data/trips_history.jsonl")
print("textFile    :", raw.getNumPartitions(), "partitions")

# 3) Transformer un RDD existant (nouveau RDD, car les RDD sont immuables)
trips = raw.map(json.loads)

print(raw.count())

parallelize : 3 partitions
textFile    : 2 partitions
50000


In [26]:
# filter (trajets terminés)


t0 = time.time()
ended = trips.filter(lambda t: t["status"] == "ENDED").cache()
print("Trajets ENDED :", ended.count())   # ACTION

Trajets ENDED : 15160


In [27]:
# map + reduceByKey (chiffre d'affaires par ville)


revenue = ended.map(lambda t: (t["city"], t["price_mad"] or 0.0)).reduceByKey(add)
for ville, ca in sorted(revenue.collect()):
    print(f"{ville:12s} {ca:12.2f} MAD")

Casablanca       98655.90 MAD
Marrakech        53879.35 MAD
Rabat            55483.53 MAD
Tanger           36678.65 MAD


In [28]:
# Durée moyenne par ville et type de véhicule


avg_dur = (ended.map(lambda t: ((t["city"], t["vehicle_type"]), (t["duration_s"], 1)))
                .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                .mapValues(lambda s: round(s[0] / s[1], 1)))
for (ville, vt), d in sorted(avg_dur.collect()):
    print(f"{ville:12s} {vt:8s} {d:8.1f} s")

Casablanca   BIKE       1192.6 s
Casablanca   EBIKE      1188.9 s
Casablanca   SCOOTER    1189.0 s
Marrakech    BIKE       1210.0 s
Marrakech    EBIKE      1201.5 s
Marrakech    SCOOTER    1212.8 s
Rabat        BIKE       1185.0 s
Rabat        EBIKE      1206.3 s
Rabat        SCOOTER    1185.4 s
Tanger       BIKE       1214.9 s
Tanger       EBIKE      1220.2 s
Tanger       SCOOTER    1156.4 s


In [29]:
# Top 5 des stations de départ

top5 = (ended.map(lambda t: (t["start_station"], 1))
             .reduceByKey(add)
             .takeOrdered(5, key=lambda x: -x[1]))
print(top5)
print("Temps total RDD : %.1f s" % (time.time() - t0))

[('S-CAS-007', 272), ('S-MAR-010', 72), ('S-CAS-141', 66), ('S-CAS-137', 64), ('S-CAS-154', 64)]
Temps total RDD : 5.2 s


In [30]:
# Lineage

print(revenue.toDebugString().decode())

(2) PythonRDD[83] at collect at /tmp/ipykernel_327/2395217010.py:5 []
 |  MapPartitionsRDD[82] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[81] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[80] at reduceByKey at /tmp/ipykernel_327/2395217010.py:4 []
    |  PythonRDD[79] at reduceByKey at /tmp/ipykernel_327/2395217010.py:4 []
    |  PythonRDD[77] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 1119.5 KiB; DiskSize: 0.0 B
    |  file:///data/trips_history.jsonl MapPartitionsRDD[75] at textFile at NativeMethodAccessorImpl.java:0 []
    |  file:///data/trips_history.jsonl HadoopRDD[74] at textFile at NativeMethodAccessorImpl.java:0 []
